Matching API
pmatch_step(e, pat) -> list[Term, Pat]

Does ematching necessarily become e-uniification or can they be kept distinct?
WCOJ style ematching


Is store/update also be seen as swap?   swap(arr, index, "a", "b")  Kind of like CAS. If it isn't "a" leave alone. If it is "a" swap out for "b" at index

Semipersistent sets? 

pushable arrays and "is-prefix-of"

Semipersistent Proof objects on destructive rewriting

semi-persistent zippers. Move up, down, replace. Hmm.


Rudi pointed out that an array to concrete strings (no non ground strings) would be simpler. i should've done that


ematching over non group union ginf



The "free" proof object
cong_f with f as a parameter. refl_t
Proof actions. A proof action is the rewrite chain it applies
Associaitvity on the nose.
type list[Proof]




Gabriel Scherer pointed me to some very interesting recent papers Snapshottable Stores <https://inria.hal.science/hal-04887939v1/document>  and <https://inria.hal.science/hal-04859464v1/document>  Snapshottable types: free, absorbing, custom    which are particularly interesting to me as having suggestions for a generic picture of semipersistent references as a library


x = x + 1 as meaning x is inf (in fixed point.) That's kind of intriguing.
Cons and the is-prefix-of union find
is-prefix-of(x, Cons(x, ))
is-post-fix-of()




In [ ]:
from dataclasses import dataclass
class Proof: ...
@dataclass
class Cong(Proof):
    f : str
    args : list[Proof]
@dataclass
class Refl(Proof):
    t : Term # Important or not? id(t) vs id. Usually the t matters.
@dataclass
class Trans(Proof): #  
    pfs : list[Proof]
@dataclass 
class Sym(Proof):
    pf : Proof

def cong(f : str, args : list[Proof]) -> Proof:
    if all(isinstance(a, Refl) for a in args):
        return Refl(App(f, [a.t for a in args]))
    elif isinstance(args[0], Sym):
        return Sym(cong(f, [sym(a) for a in args]))
    else:
        return Cong(f, args)
def comp(p1, p2) -> Proof:
    # ugly
    if isinstance(p1, Refl):
        return p2
    if isinstance(p2, Refl):
        return p1
    elif isinstance(p1, Cong) and isinstance(p2, Cong) and p1.f == p2.f:
        return Cong(p1.f, [comp(a1, a2) for a1, a2 in zip(p1.args, p2.args)])
    elif isinstance(p1, Trans):
        return Trans(p1.pfs + [p2])
    elif isinstance(p2, Trans):
        return Trans([p1] + p2.pfs)
    else:
        return Trans([p1, p2])
def sym(pf : Proof) -> Proof:
    if isinstance(pf, Refl):
        return pf
    elif isinstance(pf, Sym):
        return pf.pf
    else:
        return Sym(pf)

In [ ]:
# Rudi's code:
# We are talking about equivalent among Tables that map str to str.

type K = str
type V = int

# We choose K != V to clarify that they are incompatible to each other.

# The tuple is always ordered by K lex order.
type G = tuple[(K, V), ...]

type Id = int
type GId = tuple[G, Id]

def change(g: G, k, v) -> G:
    d = dict()
    for k2, v2 in g:
        d[k2] = v2
    d[k] = v

    out = []
    for key in sorted(list(d.keys())):
        out.append((key, d[key]))
    return tuple(out)

def keys(g: G):
    return set([k for k, v in list(g)])

def remove(g: G, k, v) -> G:
    return tuple([kv for kv in list(g) if kv != (k, v)])

# r is applied first in `l*r*x`.
def compose(l: G, r: G) -> G:
    out = r
    for k, v in l:
        out = change(out, k, v)
    return out

def coset(g, lat):
    for k, v in lat:
        g = remove(g, k, v)
    return g

# remove common assignments from g1 and g2.
# returns None, if the same key gets mapped to different value.
def sub(g1, g2) -> tuple[G, G] | None:
    for k in keys(g1) & keys(g2):
        if g1[k] != g2[k]:
            # We are in a `x["a"="c"] = y["a"="b"]` kind of situation.
            raise "contradiction!"
        v = g1[k]
        g1 = remove(g1, k, v)
        g2 = remove(g2, k, v)
    return g1, g2

class UF:
    def __init__(self):
        self.to_leader = {} # Id -> GId
        self.lattice = {} # Id -> set (K, V)

    def alloc(self):
        i = len(self.to_leader)
        self.to_leader[i] = ((), i)
        self.lattice[i] = set()
        return ((), i)

    def find(self, gx: GId) -> GId:
        g, x = gx
        while self.to_leader[x] != ((), x):
            g2, x2 = self.to_leader[x]
            g, x = compose(g, g2), x2

        g = coset(g, self.lattice[x])
        return g, x

    def union(self, gx1: GId, gx2: GId):
        g1,x1 = self.find(gx1)
        g2,x2 = self.find(gx2)
        g1, g2 = sub(g1,g2)

        if x1 == x2:
            self.lattice[x1].update(list(g1) + list(g2))
        else:
            ((), z) = self.alloc()

            self.to_leader[x1] = (g2, z)
            for k, v in self.lattice[x1]:
                if k not in keys(g1): self.lattice[z].add((k, v))
            del self.lattice[x1]

            self.to_leader[x2] = (g1, z)
            for k, v in self.lattice[x2]:
                if k not in keys(g2): self.lattice[z].add((k, v))
            del self.lattice[x2]

    def is_eq(self, x: GId, y: GId) -> bool:
        return self.find(x) == self.find(y)

uf = UF()
((), a) = uf.alloc()
((), b) = uf.alloc()

uf.union(
    ((("s3", 42),), a),
    ((), a)
)

uf.union(
    ((("s1", 7),), a),
    ((("s2", 9),), b)
)

print("a=",a)
print("b=",a)
print(uf.to_leader)
print(uf.lattice)

print(uf.is_eq(((), a), ((), b)))

print(uf.is_eq(
    ((("s1", 7),), a),
    ((("s2", 9),), b)
))

a= 0
b= 0
{0: ((('s2', 9),), 2), 1: ((('s1', 7),), 2), 2: ((), 2)}
{2: {('s3', 42)}}
False
True
